In [ ]:
import librosa
import glob
import pathlib
import os
import math
import pandas as pd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from datetime import datetime
from tqdm import tqdm
DATA_INPUT_ROOT = "../data/sampled" 
IMAGE_OUTPUT_ROOT = "./hasil/visualisasi" 

In [ ]:
# Durasi cuplikan yang diplot dalam milidetik (agar titik sampel terlihat jelas)
PLOT_WINDOW_MS = 50.0 


# ── EXACT FUNCTIONS FROM PROJECT ──────────────────────────────────────────────

def sampling(wav_path: str) -> tuple[np.ndarray, int]:
    """Membaca file wav, convert ke float64 mono."""
    samples, sr = sf.read(wav_path, dtype="float64")
    if samples.ndim == 2:       # stereo jadi mono
        samples = samples[:, 0]
    return samples, sr


def quantization(samples: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Melakukan kuantisasi 16-bit PCM."""
    # Step 1 – level rentang -32768 sampai +32767
    max_level = 32_767
    min_level = -32_768
    # Step 2 – kode PCM: amplitudo kontinu x 32767 kemudian bulatkan
    kode_pcm = np.clip(np.round(samples * max_level), min_level, max_level).astype(int)
    # Step 3 – rekonstruksi kode PCM / 32767 = amplitudo diskrit
    quantized = kode_pcm / max_level
    return kode_pcm, quantized


# ── VISUALIZATION HELPERS (Styling & Saving) ─────────────────────────────────

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def style_ax(ax):
    """Menerapkan gaya gelap (dark theme) pada axis matplotlib."""
    ax.set_facecolor("#0f0f11") # Background plot sangat gelap
    for spine in ax.spines.values():
        spine.set_edgecolor("#2a2a35") # Warna garis tepi
    ax.tick_params(colors="#666", labelsize=8) # Warna angka di axes
    ax.xaxis.label.set_color("#666") # Warna label X
    ax.yaxis.label.set_color("#666") # Warna label Y
    ax.grid(color="#1a1a22", linestyle='--', linewidth=0.5) # Grid halus


def save_plot(fig, audio_id, kind, base_dir):
    """Menyimpan figure ke folder terstruktur: base_dir/audio_id/kind/filename.png"""
    folder = os.path.join(base_dir, audio_id, kind)
    ensure_dir(folder)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{kind}_{audio_id}_{timestamp}.png"
    filepath = os.path.join(folder, filename)
    
    # Simpan dengan bbox_inches="tight" agar label tidak terpotong
    # facecolor harus diset agar background figure (bukan cuma plot) jadi gelap
    fig.savefig(filepath, dpi=150, bbox_inches="tight", facecolor="#18181b")
    plt.close(fig) # Tutup figure untuk menghemat memori


# ── MAIN PROCESSING LOOP (Struktur seperti kode terakhir user) ───────────────

def run_automated_visualization():
    print(f"[*] Memulai visualisasi otomatis...")
    print(f"[*] Input: {os.path.abspath(DATA_INPUT_ROOT)}")
    print(f"[*] Output: {os.path.abspath(IMAGE_OUTPUT_ROOT)}")

    # Mencari seluruh file .wav secara rekursif
    wav_files = glob.glob(os.path.join(DATA_INPUT_ROOT, "**/*.wav"), recursive=True)
    
    if not wav_files:
        print(f"[-] Tidak ditemukan file .wav di folder input.")
        return

    saved_count = 0
    
    # Menggunakan tqdm untuk progress bar (opsional, jika tidak punya, hapus tqdm())
    for wav_path in tqdm(wav_files, desc="Memproses Audio"):
        try:
            # Mengambil nama file tanpa ekstensi sebagai ID
            audio_id = os.path.splitext(os.path.basename(wav_path))[0]

            # 1. Proses Sampling
            samples_full, sr = sampling(wav_path)

            # 2. Proses Kuantisasi (Seluruh Sinyal)
            kode_pcm_full, quantized_full = quantization(samples_full)

            # 3. Ekstrak Cuplikan (Snippet) untuk Plotting
            # Kita cuma plot 50ms pertama agar titik sampel terlihat jelas
            num_samples_to_plot = int(PLOT_WINDOW_MS * sr / 1000)
            # Pastikan tidak melebihi panjang audio asli
            end_idx = min(num_samples_to_plot, len(samples_full))
            
            samples_snip = samples_full[:end_idx]
            quantized_snip = quantized_full[:end_idx]
            kode_pcm_snip = kode_pcm_full[:end_idx]
            # Membuat sumbu waktu dalam ms
            t_ms = np.arange(0, end_idx) / sr * 1000

            if len(samples_snip) == 0:
                print(f"[!] File audio kosong atau terlalu pendek: {audio_id}")
                continue

            # --- Visualisasi 1: SAMPLING ---
            fig_s, ax_s = plt.subplots(figsize=(10, 4))
            style_ax(ax_s)

            # Plot sinyal analog asli (garis biru tipis)
            ax_s.plot(t_ms, samples_snip, color="#3d6be0", lw=1.0, alpha=0.7, zorder=2, label='Sinyal Asli')
            ax_s.axhline(0, color="#2a2a35", lw=0.5)

            # Plot titik-titik sampel (batang dan titik oranye amber)
            # Stem plot manual agar estetika sama dengan project asli
            for xi, yi in zip(t_ms, samples_snip):
                ax_s.plot([xi, xi], [0, yi], color="#f59e0b", lw=0.7, alpha=0.5, zorder=3)
            ax_s.scatter(t_ms, samples_snip, color="#f59e0b", s=15, zorder=4, edgecolors="none", label='Sampel')

            nyquist = sr / 2
            ax_s.set_title(
                f"Sampling [{audio_id}] - Cuplikan {PLOT_WINDOW_MS}ms\n"
                f"SR: {sr:,} Hz | Sampel di Plot: {len(samples_snip):,} | Nyquist: {nyquist:,.0f} Hz",
                color="#e2e2e8", fontsize=10, pad=10
            )
            ax_s.set_xlabel("Waktu (ms)")
            ax_s.set_ylabel("Amplitudo")
            ax_s.set_xlim(t_ms[0], t_ms[-1])
            ax_s.set_ylim(-1.1, 1.1)
            # ax_s.legend(loc='upper right', fontsize=8, labelcolor='#aaa', frameon=False)

            # Simpan Plot Sampling
            save_plot(fig_s, audio_id, "sampling", IMAGE_OUTPUT_ROOT)


            # --- Visualisasi 2: QUANTIZATION ---
            fig_q, ax_q = plt.subplots(figsize=(10, 4))
            style_ax(ax_q)

            # Plot sinyal asli sebagai background (biru sangat transparan)
            ax_q.plot(t_ms, samples_snip, color="#3d6be0", lw=1.0, alpha=0.2, zorder=2)
            ax_q.axhline(0, color="#2a2a35", lw=0.5)

            # Plot tangga kuantisasi (garis hijau)
            # Menggunakan drawstyle='steps-post' untuk membuat efek tangga
            ax_q.plot(t_ms, quantized_snip, color="#22c77e", lw=1.5, drawstyle='steps-post', zorder=4, label='Sinyal Terkuantisasi')
            # Plot titik-titik di setiap langkah tangga
            ax_q.scatter(t_ms, quantized_snip, color="#22c77e", s=10, zorder=5, edgecolors="none")

            snr_theo = round(6.02 * 16 + 1.76, 1) # Teoretis 16-bit
            pcm_min = kode_pcm_snip.min()
            pcm_max = kode_pcm_snip.max()

            ax_q.set_title(
                f"Quantization (16-bit PCM) [{audio_id}] - Cuplikan {PLOT_WINDOW_MS}ms\n"
                f"Levels: 65,536 | SNR Teoretis: ~{snr_theo} dB | Rentang PCM di Plot: [{pcm_min}, {pcm_max}]",
                color="#e2e2e8", fontsize=10, pad=10
            )
            ax_q.set_xlabel("Waktu (ms)")
            ax_q.set_ylabel("Amplitudo Terdiskrit")
            ax_q.set_xlim(t_ms[0], t_ms[-1])
            ax_q.set_ylim(-1.1, 1.1)
            # ax_q.legend(loc='upper right', fontsize=8, labelcolor='#aaa', frameon=False)

            # Simpan Plot Kuantisasi
            save_plot(fig_q, audio_id, "quantization", IMAGE_OUTPUT_ROOT)

            saved_count += 1

        except Exception as e:
            # Error handling seperti struktur kode user
            print(f"[!] Error pada file : {wav_path} : {e}")

    print("-" * 30)
    print(f"[*] Selesai!")
    print(f"[*] Total Audio Diproses : {saved_count}")
    print(f"[*] Total Gambar Disimpan : {saved_count * 2}") # Karena tiap audio buat 2 gambar



In [ ]:
run_automated_visualization()